#**Data Preprocessing**

In [ ]:
# Set your Kaggle API key as an environment variable
import os
os.environ['KAGGLE_USERNAME'] = ""
os.environ['KAGGLE_KEY'] = ""

In [ ]:
import kaggle
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
kaggle.api.authenticate()

In [ ]:
api.dataset_download_files(
  'muhammadhananasghar/human-emotions-datasethes',
  path='/content',
  unzip=True
)

Dataset URL: https://www.kaggle.com/datasets/muhammadhananasghar/human-emotions-datasethes


In [ ]:
os.system('rm -rf /content/EmotionsDataset')
os.system('rm -rf /content/EmotionsDataset_Splitted')
os.system('rm -rf /content/sample_data')

0

In [ ]:
train_dir = '/content/Emotions Dataset/Emotions Dataset/train'
test_dir = '/content/Emotions Dataset/Emotions Dataset/test'

In [ ]:
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
class_names = ['angry', 'happy', 'sad']

In [ ]:
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
  train_dir,
  labels='inferred',
  label_mode='int',
  class_names=class_names,
  color_mode='rgb',
  batch_size=None,
  image_size=(256, 256),
  shuffle=True,
  seed=101,
  data_format='channels_last'
)

Found 6799 files belonging to 3 classes.


In [ ]:
val_dataset, test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
  test_dir,
  labels='inferred',
  label_mode='int',
  class_names=class_names,
  color_mode='rgb',
  image_size=(256, 256),
  batch_size=None,
  shuffle=True,
  seed=101,
  validation_split=0.5,
  subset='both',
  data_format='channels_last'
)

Found 2278 files belonging to 3 classes.
Using 1139 files for training.
Using 1139 files for validation.


In [ ]:
rescale_layer = tf.keras.layers.Rescaling(1/255)

def rescale(image, label):
    image = rescale_layer(image)
    return image, label

train_dataset = train_dataset.map(rescale)
val_dataset = val_dataset.map(rescale)
test_dataset = test_dataset.map(rescale)

In [ ]:
train_dataset = train_dataset.batch(32).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(1).prefetch(tf.data.AUTOTUNE)

#**CallBacks**

In [ ]:
#Saves the most with best performance among all epochs
checkpoint_path = "/content/best_model.keras"
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_sparse_categorical_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1
)

In [ ]:
#Reduces learning rate when there is no increase in accuracy
learning_rate_callback = tf.keras.callbacks.ReduceLROnPlateau(
  monitor='val_sparse_categorical_accuracy',
  factor=0.3,
  patience=3,
  min_lr=1e-6,
  min_delta=0.01,
)

#**Baseline Model**

##Architecture

In [ ]:
from keras.layers import Normalization, Dense, Input, Conv2D, MaxPool2D, Flatten, BatchNormalization

model = tf.keras.Sequential([
    Input(shape = (256, 256, 3)),

    Conv2D(filters = 4, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Conv2D(filters = 16, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Conv2D(filters = 64, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Conv2D(filters = 64, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Conv2D(filters = 128, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Flatten(),

    Dense(1000, activation = "relu"),
    BatchNormalization(),

    Dense(250, activation = "relu"),
    BatchNormalization(),

    Dense(3, activation = "softmax")
])

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 254, 254, 4)         │             108 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 254, 254, 4)         │              16 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 127, 127, 4)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 125, 125, 16)        │             576 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 125, 125, 16)        │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 62, 62, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 60, 60, 64)          │           9,216 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 60, 60, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 28, 28, 64)          │          36,864 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 28, 28, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 12, 12, 128)         │          73,728 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 12, 12, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 6, 6, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 4608)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1000)                │       4,609,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 1000)                │           4,000 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 4,986,599 (19.02 MB)

 Trainable params: 4,983,547 (19.01 MB)

 Non-trainable params: 3,052 (11.92 KB)

##Model Training

0.001 is the best starting learning rate and 15 epochs is the best epoch size.

In [ ]:
from keras.losses import SparseCategoricalCrossentropy
from keras.optimizers import Adam
from keras.metrics import SparseCategoricalAccuracy

model.compile(
  optimizer=Adam(learning_rate=0.001),
  loss=SparseCategoricalCrossentropy(),
  metrics=[SparseCategoricalAccuracy()]
)

In [ ]:
# For learning rate = 0.001
history = model.fit(train_dataset, validation_data = val_dataset, epochs =15, verbose = 1, callbacks = [checkpoint_callback, learning_rate_callback])

Epoch 1/20
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 1.1214 - sparse_categorical_accuracy: 0.5524
Epoch 1: val_sparse_categorical_accuracy improved from -inf to 0.44864, saving model to /content/best_model.keras
213/213 ━━━━━━━━━━━━━━━━━━━━ 28s 84ms/step - loss: 1.1205 - sparse_categorical_accuracy: 0.5526 - val_loss: 2.8717 - val_sparse_categorical_accuracy: 0.4486 - learning_rate: 0.0010
Epoch 2/20
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.7100 - sparse_categorical_accuracy: 0.6914
Epoch 2: val_sparse_categorical_accuracy improved from 0.44864 to 0.49693, saving model to /content/best_model.keras
213/213 ━━━━━━━━━━━━━━━━━━━━ 27s 51ms/step - loss: 0.7097 - sparse_categorical_accuracy: 0.6916 - val_loss: 2.0887 - val_sparse_categorical_accuracy: 0.4969 - learning_rate: 0.0010
Epoch 3/20
212/213 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 0.5544 - sparse_categorical_accuracy: 0.7705
Epoch 3: val_sparse_categorical_accuracy improved from 0.49693 to 0.75417, saving model t

In [ ]:
model.evaluate(test_dataset)

1139/1139 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.6763 - sparse_categorical_accuracy: 0.8348


[0.6919892430305481, 0.8270412683486938]

#**Hyperparameter Tuned Model**

##Architecture

In [ ]:
#This architecture is hyperparameter tuned on validation dataset.
from keras.layers import Normalization, Dense, Input, Conv2D, MaxPool2D, Flatten, BatchNormalization

model = tf.keras.Sequential([
    Input(shape = (256, 256, 3)),

    Conv2D(filters = 8, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Conv2D(filters = 16, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Conv2D(filters = 64, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Conv2D(filters = 32, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Conv2D(filters = 32, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Flatten(),

    Dense(1000, activation = "relu"),
    BatchNormalization(),

    Dense(500, activation = "relu"),
    BatchNormalization(),

    Dense(50, activation = "relu"),
    BatchNormalization(),

    Dense(10, activation = "relu"),
    BatchNormalization(),

    Dense(3, activation = "softmax")
])

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 254, 254, 8)         │             216 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 254, 254, 8)         │              32 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 127, 127, 8)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 125, 125, 16)        │           1,152 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 125, 125, 16)        │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 62, 62, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 60, 60, 64)          │           9,216 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 60, 60, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 28, 28, 32)          │          18,432 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 28, 28, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 14, 14, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 12, 12, 32)          │           9,216 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 12, 12, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 6, 6, 32)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 1152)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1000)                │       1,153,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 1000)                │           4,000 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 1,724,173 (6.58 MB)

 Trainable params: 1,720,749 (6.56 MB)

 Non-trainable params: 3,424 (13.38 KB)

##Model Training

In [ ]:
from keras.losses import SparseCategoricalCrossentropy
from keras.optimizers import Adam
from keras.metrics import SparseCategoricalAccuracy
model.compile(
  optimizer=Adam(learning_rate=0.001),
  loss=SparseCategoricalCrossentropy(),
  metrics=[SparseCategoricalAccuracy()]
)

In [ ]:
# For learning rate = 0.001
history = model.fit(train_dataset, validation_data = val_dataset, epochs =15, verbose = 1, callbacks = [checkpoint_callback, learning_rate_callback])

Epoch 1/15
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 0.9874 - sparse_categorical_accuracy: 0.5269
Epoch 1: val_sparse_categorical_accuracy improved from -inf to 0.44864, saving model to /content/best_model.keras
213/213 ━━━━━━━━━━━━━━━━━━━━ 33s 91ms/step - loss: 0.9868 - sparse_categorical_accuracy: 0.5272 - val_loss: 1.6805 - val_sparse_categorical_accuracy: 0.4486 - learning_rate: 0.0010
Epoch 2/15
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 0.7379 - sparse_categorical_accuracy: 0.6729
Epoch 2: val_sparse_categorical_accuracy improved from 0.44864 to 0.61194, saving model to /content/best_model.keras
213/213 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - loss: 0.7376 - sparse_categorical_accuracy: 0.6730 - val_loss: 0.8543 - val_sparse_categorical_accuracy: 0.6119 - learning_rate: 0.0010
Epoch 3/15
212/213 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 0.5806 - sparse_categorical_accuracy: 0.7512
Epoch 3: val_sparse_categorical_accuracy improved from 0.61194 to 0.69886, saving model to

In [ ]:
model.evaluate(test_dataset)

1139/1139 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.6633 - sparse_categorical_accuracy: 0.8148


[0.6312380433082581, 0.8165057301521301]